# Занятие 3_3_2. Инструменты для сбора данных. Пример 2

## Получение данных через `requests`, `httpx` и Beautiful Soup

В этом примере мы создадим небольшой локальный учебный сайт. Затем:

1. получим JSON через `requests`;
2. получим второй JSON через `httpx`;
3. загрузим HTML-страницу;
4. извлечём данные из HTML с помощью Beautiful Soup;
5. преобразуем результаты в таблицы pandas;
6. объединим собранные данные;
7. сохраним итог в CSV.

Все обращения выполняются к адресу `127.0.0.1`, поэтому интернет не требуется.

## Что получится в результате

В папке `результаты_пример_2` будет создан файл:

`собранный_каталог.csv`

В нём будут объединены:

- сведения о товарах из первого JSON;
- остатки из второго JSON;
- цены из HTML-страницы.

## Подготовка рабочей среды

Запустите следующие две ячейки **до всех остальных**.

Первая ячейка проверяет наличие библиотек и устанавливает только отсутствующие пакеты. Вторая ячейка выполняет все импорты. После этого notebook нужно запускать сверху вниз командой **«Выполнить все»**.

Если компьютер работает без доступа к интернету, библиотеки нужно заранее установить из файла `requirements.txt` командой:

```bash
python -m pip install -r requirements.txt
```


In [ ]:
# Подключаем стандартные модули Python для проверки и установки библиотек.
import importlib.util
import subprocess
import sys

# Сопоставляем имя модуля в Python и имя пакета для установки через pip.
необходимые_пакеты = {
    "pandas": "pandas>=2.0,<4",
    "requests": "requests>=2.31,<3",
    "httpx": "httpx>=0.27,<1",
    "bs4": "beautifulsoup4>=4.12,<5",
}

# Создаём пустой список для пакетов, которых нет в текущем окружении.
отсутствующие_пакеты = []

# Проверяем каждый модуль по очереди.
for имя_модуля, пакет_для_установки in необходимые_пакеты.items():
    # find_spec возвращает None, если модуль не найден.
    if importlib.util.find_spec(имя_модуля) is None:
        # Добавляем отсутствующий пакет в список установки.
        отсутствующие_пакеты.append(пакет_для_установки)

# Устанавливаем только те пакеты, которых не хватает.
if отсутствующие_пакеты:
    # Показываем, какие пакеты будут установлены.
    print("Будут установлены пакеты:", отсутствующие_пакеты)

    try:
        # Запускаем pip через тот же Python, который использует текущий notebook.
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--quiet",
                *отсутствующие_пакеты,
            ]
        )
        # Сообщаем об успешном завершении установки.
        print("Установка завершена. Можно выполнять следующую ячейку.")

    except subprocess.CalledProcessError as ошибка:
        # Формируем понятное сообщение, если pip не смог установить библиотеки.
        raise RuntimeError(
            "Не удалось установить библиотеки. "
            "Проверьте доступ к интернету или заранее выполните команду "
            "python -m pip install -r requirements.txt"
        ) from ошибка
else:
    # Сообщаем, что все библиотеки уже доступны.
    print("Все необходимые библиотеки уже установлены.")


In [ ]:
# Подключаем Path для работы с папками и файлами.
from pathlib import Path

# Подключаем json для чтения и сохранения JSON-файлов.
import json

# Подключаем threading для запуска локального сервера в отдельном потоке.
import threading

# Подключаем встроенный HTTP-сервер Python.
from http.server import ThreadingHTTPServer, SimpleHTTPRequestHandler

# Подключаем partial, чтобы передать серверу нужную папку.
from functools import partial

# Подключаем pandas для работы с таблицами.
import pandas as pd

# Подключаем requests для выполнения HTTP-запросов.
import requests

# Подключаем httpx как второй HTTP-клиент.
import httpx

# Подключаем пакет bs4, чтобы вывести его версию.
import bs4

# Подключаем BeautifulSoup для разбора HTML.
from bs4 import BeautifulSoup

# Явно подключаем display для отображения DataFrame в notebook.
from IPython.display import display

# Показываем версии основных библиотек.
print("pandas:", pd.__version__)
print("requests:", requests.__version__)
print("httpx:", httpx.__version__)
print("beautifulsoup4:", bs4.__version__)

# Сообщаем, что импорт завершён успешно.
print("Все библиотеки успешно импортированы.")


## 1. Создаём рабочие папки

Папка `учебный_сайт_пример_2` будет играть роль небольшого сайта.
Папка `результаты_пример_2` будет хранить итоговые файлы.

In [ ]:
# Получаем папку, из которой запущен notebook.
рабочая_папка = Path.cwd()

# Создаём путь к папке учебного сайта.
папка_сайта = рабочая_папка / "учебный_сайт_пример_2"

# Создаём путь к папке API внутри учебного сайта.
папка_api = папка_сайта / "api"

# Создаём путь к папке результатов.
папка_результатов = рабочая_папка / "результаты_пример_2"

# Создаём папку сайта, если её ещё нет.
папка_сайта.mkdir(exist_ok=True)

# Создаём папку API, включая родительскую папку.
папка_api.mkdir(parents=True, exist_ok=True)

# Создаём папку результатов.
папка_результатов.mkdir(exist_ok=True)

# Показываем созданные пути.
print("Папка сайта:", папка_сайта)
print("Папка API:", папка_api)
print("Папка результатов:", папка_результатов)

## 2. Создаём первый JSON-источник

Первый источник содержит основные сведения о товарах.

In [ ]:
# Создаём словарь с данными первого учебного API.
данные_товаров = {
    "товары": [
        {
            "Код товара": "Т-1",
            "Название товара": "Мышь",
            "Категория": "Аксессуары"
        },
        {
            "Код товара": "Т-2",
            "Название товара": "Клавиатура",
            "Категория": "Аксессуары"
        },
        {
            "Код товара": "Т-3",
            "Название товара": "Наушники",
            "Категория": "Аудио"
        }
    ]
}

# Создаём путь к первому JSON-файлу.
путь_товаров_json = папка_api / "товары.json"

# Открываем файл для записи в кодировке UTF-8.
with путь_товаров_json.open("w", encoding="utf-8") as файл:
    # Сохраняем словарь в JSON с читаемыми русскими символами.
    json.dump(данные_товаров, файл, ensure_ascii=False, indent=2)

# Сообщаем, где создан файл.
print("Создан файл:", путь_товаров_json)

## 3. Создаём второй JSON-источник

Второй источник содержит сведения об остатках.

In [ ]:
# Создаём словарь с данными второго учебного API.
данные_остатков = {
    "остатки": [
        {
            "Код товара": "Т-1",
            "Остаток": 12,
            "Статус наличия": "В наличии"
        },
        {
            "Код товара": "Т-2",
            "Остаток": 4,
            "Статус наличия": "Мало"
        },
        {
            "Код товара": "Т-3",
            "Остаток": 0,
            "Статус наличия": "Нет в наличии"
        }
    ]
}

# Создаём путь ко второму JSON-файлу.
путь_остатков_json = папка_api / "остатки.json"

# Открываем файл для записи.
with путь_остатков_json.open("w", encoding="utf-8") as файл:
    # Сохраняем словарь в JSON.
    json.dump(данные_остатков, файл, ensure_ascii=False, indent=2)

# Показываем путь к файлу.
print("Создан файл:", путь_остатков_json)

## 4. Создаём HTML-страницу

HTML-страница содержит цены товаров. Классы HTML-элементов записаны латиницей, а значения и будущие столбцы — по-русски.

In [ ]:
# Создаём HTML-код как многострочную строку.
html_страница = """
<!DOCTYPE html>
<html lang="ru">
<head>
    <meta charset="UTF-8">
    <title>Учебный каталог</title>
</head>
<body>
    <h1 id="store-name">Магазин «Учебная техника»</h1>
    <p id="updated-at">Обновлено: 17.07.2026</p>

    <div class="product">
        <span class="code">Т-1</span>
        <span class="price">2500</span>
    </div>

    <div class="product">
        <span class="code">Т-2</span>
        <span class="price">4200</span>
    </div>

    <div class="product">
        <span class="code">Т-3</span>
        <span class="price">5600</span>
    </div>
</body>
</html>
"""

# Создаём путь к HTML-файлу.
путь_html = папка_сайта / "каталог.html"

# Сохраняем HTML-код в файл.
путь_html.write_text(html_страница, encoding="utf-8")

# Показываем путь к созданной странице.
print("Создан файл:", путь_html)

## 5. Запускаем локальный HTTP-сервер

Эта ячейка техническая. Она позволяет обращаться к созданным файлам так же, как к небольшому сайту.
Запоминать всю конструкцию сервера не требуется.

In [ ]:
# Создаём обработчик запросов, который будет отдавать файлы из папки сайта.
обработчик = partial(SimpleHTTPRequestHandler, directory=str(папка_сайта))

# Создаём сервер на локальном адресе.
# Значение 0 означает, что Python самостоятельно выберет свободный порт.
локальный_сервер = ThreadingHTTPServer(("127.0.0.1", 0), обработчик)

# Получаем выбранный сервером номер порта.
номер_порта = локальный_сервер.server_address[1]

# Создаём отдельный поток для работы сервера.
поток_сервера = threading.Thread(
    target=локальный_сервер.serve_forever,
    daemon=True
)

# Запускаем поток сервера.
поток_сервера.start()

# Формируем базовый адрес локального сайта.
базовый_адрес = f"http://127.0.0.1:{номер_порта}"

# Показываем адрес сайта.
print("Локальный сайт запущен:", базовый_адрес)

## 6. Получаем первый JSON через `requests`

`requests.get()` отправляет GET-запрос и возвращает объект ответа.

In [ ]:
# Формируем адрес первого JSON-источника.
адрес_товаров = f"{базовый_адрес}/api/товары.json"

# Отправляем GET-запрос.
ответ_requests = requests.get(
    адрес_товаров,
    timeout=10
)

# Проверяем, что сервер не вернул HTTP-ошибку.
ответ_requests.raise_for_status()

# Преобразуем JSON-ответ в словарь Python.
json_товаров = ответ_requests.json()

# Показываем HTTP-статус.
print("HTTP-статус:", ответ_requests.status_code)

# Показываем полученный словарь.
print(json_товаров)

In [ ]:
# Получаем список товаров из словаря.
список_товаров = json_товаров["товары"]

# Преобразуем список словарей в DataFrame.
таблица_товаров = pd.DataFrame(список_товаров)

# Показываем таблицу.
display(таблица_товаров)

## 7. Получаем второй JSON через `httpx`

В базовом синхронном сценарии `httpx` используется очень похоже на `requests`.

In [ ]:
# Формируем адрес второго JSON-источника.
адрес_остатков = f"{базовый_адрес}/api/остатки.json"

# Отправляем GET-запрос через httpx.
ответ_httpx = httpx.get(
    адрес_остатков,
    timeout=10.0
)

# Проверяем, что запрос завершился без HTTP-ошибки.
ответ_httpx.raise_for_status()

# Преобразуем JSON-ответ в словарь Python.
json_остатков = ответ_httpx.json()

# Показываем HTTP-статус.
print("HTTP-статус:", ответ_httpx.status_code)

# Показываем полученный словарь.
print(json_остатков)

In [ ]:
# Получаем список остатков из словаря.
список_остатков = json_остатков["остатки"]

# Преобразуем список в DataFrame.
таблица_остатков = pd.DataFrame(список_остатков)

# Показываем таблицу остатков.
display(таблица_остатков)

## 8. Получаем HTML-страницу

Для загрузки HTML снова используем `requests`.

In [ ]:
# Формируем адрес HTML-страницы.
адрес_html = f"{базовый_адрес}/каталог.html"

# Загружаем HTML-страницу.
ответ_html = requests.get(
    адрес_html,
    timeout=10
)

# Проверяем HTTP-статус.
ответ_html.raise_for_status()

# Явно указываем кодировку страницы.
ответ_html.encoding = "utf-8"

# Сохраняем HTML-текст в переменную.
html_текст = ответ_html.text

# Показываем первые 200 символов страницы.
print(html_текст[:200])

## 9. Разбираем HTML с помощью Beautiful Soup

Сначала создаём объект BeautifulSoup, затем ищем отдельные элементы страницы.

In [ ]:
# Создаём объект BeautifulSoup на основе HTML-текста.
страница = BeautifulSoup(
    html_текст,
    "html.parser"
)

# Находим заголовок магазина по идентификатору id.
элемент_магазина = страница.find(
    "h1",
    id="store-name"
)

# Получаем текст найденного элемента.
название_магазина = элемент_магазина.get_text(strip=True)

# Находим дату обновления.
элемент_даты = страница.find(
    "p",
    id="updated-at"
)

# Получаем текст даты обновления.
дата_обновления = элемент_даты.get_text(strip=True)

# Показываем сведения о странице.
print("Магазин:", название_магазина)
print("Дата обновления:", дата_обновления)

In [ ]:
# Находим все карточки товаров.
карточки = страница.find_all(
    "div",
    class_="product"
)

# Создаём пустой список для собранных цен.
собранные_цены = []

# Последовательно обрабатываем каждую карточку.
for карточка in карточки:
    # Получаем код товара.
    код_товара = карточка.find(
        "span",
        class_="code"
    ).get_text(strip=True)

    # Получаем цену товара как текст.
    цена_текстом = карточка.find(
        "span",
        class_="price"
    ).get_text(strip=True)

    # Преобразуем цену из текста в целое число.
    цена = int(цена_текстом)

    # Добавляем данные одной карточки в список.
    собранные_цены.append({
        "Код товара": код_товара,
        "Цена": цена
    })

# Преобразуем список словарей в DataFrame.
таблица_цен = pd.DataFrame(собранные_цены)

# Показываем полученную таблицу.
display(таблица_цен)

## 10. Объединяем собранные данные

Общий ключ во всех трёх таблицах — столбец `Код товара`.

In [ ]:
# Присоединяем остатки к основной таблице товаров.
собранный_каталог = таблица_товаров.merge(
    таблица_остатков,
    on="Код товара",
    how="left"
)

# Присоединяем цены из HTML.
собранный_каталог = собранный_каталог.merge(
    таблица_цен,
    on="Код товара",
    how="left"
)

# Добавляем название источника HTML.
собранный_каталог["Источник цены"] = название_магазина

# Показываем итоговую таблицу.
display(собранный_каталог)

## 11. Проверяем результат

Это базовая проверка процесса сбора, а не аналитический анализ.

In [ ]:
# Показываем количество строк в итоговой таблице.
print("Количество строк:", len(собранный_каталог))

# Показываем названия столбцов.
print("Столбцы:", собранный_каталог.columns.tolist())

# Считаем пропуски по каждому столбцу.
print("Пропуски:")
print(собранный_каталог.isna().sum())

# Проверяем ожидаемое количество товаров.
assert len(собранный_каталог) == 3, "Ожидалось три товара"

# Проверяем, что цены были получены для всех товаров.
assert собранный_каталог["Цена"].notna().all(), "Не для всех товаров найдена цена"

# Сообщаем об успешной проверке.
print("Проверка завершена успешно.")

## 12. Сохраняем итоговый CSV

In [ ]:
# Создаём путь к итоговому CSV-файлу.
путь_результата = папка_результатов / "собранный_каталог.csv"

# Сохраняем таблицу в CSV.
собранный_каталог.to_csv(
    путь_результата,
    index=False,
    encoding="utf-8-sig"
)

# Повторно загружаем сохранённый файл.
проверка_файла = pd.read_csv(путь_результата)

# Показываем сохранённый результат.
display(проверка_файла)

# Показываем путь к файлу.
print("Файл сохранён:", путь_результата)

## 13. Останавливаем локальный сервер

После завершения работы освобождаем занятый порт.

In [ ]:
# Останавливаем обработку новых запросов.
локальный_сервер.shutdown()

# Закрываем серверный сокет.
локальный_сервер.server_close()

# Сообщаем о завершении работы сервера.
print("Локальный сервер остановлен.")

## Итоги примера

Мы использовали:

- `requests` — первый JSON и HTML;
- `httpx` — второй JSON;
- Beautiful Soup — извлечение данных из HTML;
- pandas — создание и объединение таблиц;
- CSV — сохранение результата.

### Контрольные вопросы

1. Что возвращает `requests.get()`?
2. Для чего вызывают `raise_for_status()`?
3. Чем JSON-словарь отличается от DataFrame?
4. Для чего нужен `find_all()`?
5. По какому столбцу были объединены таблицы?